# Model Reverse Engineering
An attacker systematically queries a public AI API, collects input-output pairs, and trains a surrogate model that mimics the original — no access to weights or training data required. The cloned model can then be used for IP theft, bypassing security defenses, or reconstructing sensitive training data via model inversion.

## The Classifier
We target a penguin species classifier (Adélie vs Gentoo) exposed via a public API, querying it with flipper length and body mass inputs to collect labeled responses. Since the training data is assumed private, we reverse engineer it by systematically querying the API and training a surrogate model on the collected input-output pairs.

## Sampling Datapoints
To reverse engineer the model, randomly generate input pairs (flipper length, body mass) within realistic biological ranges, query the API for each pair to get labeled outputs, then train a surrogate classifier on those collected input-output pairs. Using informed boundaries (150-250mm, 2500-6500g) improves data quality, reducing the number of queries needed to build an accurate clone. Without meaningful boundaries, you'd need significantly more samples to compensate for the noise.

In [2]:
N_SAMPLES = 100

MIN_FLIPPER_LENGTH = 150
MAX_FLIPPER_LENGTH = 250

MIN_BODY_MASS = 2500
MAX_BODY_MASS = 6500

# Replace the IP address with the spawned instance from the lab
CLASSIFIER_URL = "http://154.57.164.74:37879/"

This generates random flipper length and body mass values spread evenly across the defined ranges to use as query inputs for the API.

In [3]:
import random
import pandas as pd

samples = {
    "Flipper Length (mm)": [],
    "Body Mass (g)": []
}

for i in range(N_SAMPLES):
    samples["Flipper Length (mm)"].append(random.uniform(MIN_FLIPPER_LENGTH, MAX_FLIPPER_LENGTH))
    samples["Body Mass (g)"].append(random.uniform(MIN_BODY_MASS, MAX_BODY_MASS))

samples_df = pd.DataFrame(samples)
print(samples_df.head())

   Flipper Length (mm)  Body Mass (g)
0           182.793401    4463.037351
1           191.501883    4361.287099
2           241.424848    6481.616689
3           226.420613    3539.040564
4           205.812522    3725.581055


Query the classifier API for each generated input pair and store the returned predicted class alongside it. This builds the labeled dataset needed to train the surrogate model.

In [4]:
import requests
import json

predictions = {"species": []}

for i in range(N_SAMPLES):
    sample = {
                "flipper_length": samples["Flipper Length (mm)"][i],
                "body_mass": samples["Body Mass (g)"][i]
            }

    prediction = json.loads(requests.get(CLASSIFIER_URL, params=sample).text).get("result")
    predictions["species"].append(prediction)

predictions_df = pd.DataFrame(predictions)
print(predictions_df.head())

  species
0  Adelie
1  Adelie
2  Gentoo
3  Gentoo
4  Adelie


## Replicating the Model
Since the original model's architecture is unknown, we choose any suitable architecture for the task — here a Logistic Regression classifier for binary classification. An exact replica isn't needed; the surrogate just needs to approximate the original's decision boundary well enough to be useful.

In [7]:
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
import joblib

# Build pipeline: normalize features first, then train logistic regression classifier
surrogate_model = make_pipeline(StandardScaler(), LogisticRegression())
# Train surrogate on API-collected input-output pairs
surrogate_model.fit(samples_df, predictions_df)

# save classifier to a file
joblib.dump(surrogate_model, 'surrogate.joblib')

C:\Users\moham\miniconda3\envs\htb-ai\Lib\site-packages\sklearn\utils\validation.py:1365: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)


['surrogate.joblib']

## Evaluation
To submit the surrogate model to the lab, we can upload it to the lab's /model endpoint.

In [8]:
with open('surrogate.joblib', 'rb') as f:
    file = f.read()

r = requests.post(CLASSIFIER_URL + '/model', files={'file': ('surrogate.joblib', file)})

print(json.loads(r.text))

{'accuracy': 0.9854014598540146, 'flag': 'HTB{ff08c0bb37e16f30a0804053a4de70ed}'}


## Mitigations
Mitigation is difficult since attacker queries are indistinguishable from legitimate use, making outright blocking impractical. Rate limiting is the primary defense — throttling query volume slows or prevents surrogate training, but must be tuned carefully to avoid impacting real users.